# LeetCode 3883 — Count Non Decreasing Arrays With Given Digit Sums bằng Python

- Link: https://leetcode.com/problems/count-non-decreasing-arrays-with-given-digit-sums/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động đếm dãy không giảm, tổng tiền tố, hai con trỏ trên danh sách ứng viên đã sắp, tiền xử lý tổng chữ số, modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countArrays(self, digitSum: list[int]) -> int:
```

## 1. Mục tiêu học

1. Tiền xử lý **danh sách ứng viên** cho mỗi tổng chữ số `s` (các số trong `[0, 5000]` có tổng chữ số `s`), đã sắp tăng.
2. Viết DP `dp_i[v]` = số mảng hợp lệ độ dài `i+1` kết thúc bằng `v`, với chuyển trạng thái `dp_i[v] = Σ_{u ≤ v} dp_{i-1}[u]`.
3. Tăng tốc phép tổng `Σ_{u ≤ v}` bằng **tổng tiền tố** (trên toàn trục giá trị) hoặc **hai con trỏ** (chỉ trên ứng viên).
4. Đánh giá kích thước: `n ≤ 1000`, mỗi tổng chữ số có tối đa vài trăm ứng viên ⇒ `O(Σ |ứng viên|)` cực nhỏ.
5. Kiểm thử với vét cạn mọi tổ hợp trên đầu vào nhỏ.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng `digitSum` độ dài `n`. Mảng `arr` độ dài `n` **hợp lệ** nếu:

- `0 ≤ arr[i] ≤ 5000`;
- `arr` **không giảm**;
- tổng chữ số của `arr[i]` bằng `digitSum[i]`.

Đếm số mảng hợp lệ, modulo `10^9 + 7`.

**Ví dụ 1:** `digitSum = [25,1]` → `6`. Phần tử sau có tổng chữ số 1 nên thuộc `{1, 10, 100, 1000}`; phần tử trước có tổng chữ số 25 và phải `≤ 1000` ⇒ chỉ có `799, 889, 898, 979, 988, 997`, mỗi số ghép với `1000`.

**Ví dụ 2:** `digitSum = [1]` → `4` (`1, 10, 100, 1000`).

**Ví dụ 3:** `digitSum = [2,49,23]` → `0` (không số nào trong `[0, 5000]` có tổng chữ số 49).

**Ràng buộc:** `1 ≤ n ≤ 1000`, `0 ≤ digitSum[i] ≤ 50`.

In [1]:
from itertools import accumulate, product
from bisect import bisect_right
import random, time

MOD = 10**9 + 7
LIMIT = 5000
examples = [([25, 1], 6), ([1], 4), ([2, 49, 23], 0)]
for ds, exp in examples:
    print(ds, "-> kỳ vọng", exp)

[25, 1] -> kỳ vọng 6
[1] -> kỳ vọng 4
[2, 49, 23] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1. Tổng chữ số và nhóm số theo tổng chữ số

`digit_sum(x) = sum(map(int, str(x)))`. Gom mọi `x ∈ [0, 5000]` vào `cand[s]` theo tổng chữ số `s` (duyệt `x` tăng dần ⇒ mỗi danh sách tự động sắp tăng).

In [2]:
def digit_sum(x):
    s = 0
    while x:
        s += x % 10
        x //= 10
    return s

cand = [[] for _ in range(51)]
for x in range(LIMIT + 1):
    cand[digit_sum(x)].append(x)
print("tổng 1:", cand[1])
print("tổng 25:", len(cand[25]), "số, 10 số đầu:", cand[25][:10], "... lớn nhất:", cand[25][-1])
print("tổng lớn nhất có thể:", max(s for s in range(51) if cand[s]), "| số ứng viên nhiều nhất:", max(map(len, cand)))

tổng 1: [1, 10, 100, 1000]
tổng 25: 80 số, 10 số đầu: [799, 889, 898, 979, 988, 997, 1699, 1789, 1798, 1879] ... lớn nhất: 4993
tổng lớn nhất có thể: 31 | số ứng viên nhiều nhất: 365


Thấy ngay: với ví dụ 1, số tổng chữ số 25 **lớn nhất** là `4993` > `1000`. Nhưng phần tử sau phải có tổng chữ số 1 và **≥** phần tử trước; các số tổng 1 chỉ có `1, 10, 100, 1000`, nên phần tử trước phải `≤ 1000` ⇒ chỉ còn `799, 889, 898, 979, 988, 997` ⇒ 6 mảng.

**Nhận xét về miền giá trị.** Số lớn nhất trong `[0, 5000]` có tổng chữ số lớn nhất là `4999` (tổng `31`). Vậy mọi `digitSum[i] ≥ 32` (trong ràng buộc tới 50) đều **không có ứng viên** ⇒ đáp án `0` ngay. Phân bố số ứng viên theo tổng chữ số có dạng "quả chuông": nhiều nhất ở khoảng giữa (~ tổng 13–18) với vài trăm số, ít ở hai đầu. Đây là cơ sở để đánh giá độ phức tạp của Cách 4.

In [3]:
sizes = [len(cand[s]) for s in range(51)]
print("số ứng viên theo tổng chữ số 0..35:")
for s in range(0, 36, 6):
    print("  " + "  ".join(f"{t:>2}:{sizes[t]:>3}" for t in range(s, min(s + 6, 36))))
print("tổng mọi ứng viên =", sum(sizes), "(= 5001 số)")

số ứng viên theo tổng chữ số 0..35:
   0:  1   1:  4   2: 10   3: 20   4: 35   5: 56
   6: 80   7:110   8:145   9:185  10:227  11:268
  12:305  13:335  14:355  15:365  16:365  17:355
  18:335  19:305  20:268  21:227  22:185  23:145
  24:110  25: 80  26: 55  27: 35  28: 20  29: 10
  30:  4  31:  1  32:  0  33:  0  34:  0  35:  0
tổng mọi ứng viên = 5001 (= 5001 số)


### 3.2. Tổng tiền tố với `itertools.accumulate`

`pre = list(accumulate(dp, initial=0))` ⇒ `Σ_{u ≤ v} dp[u] = pre[v + 1]`.

In [4]:
dp = [0, 2, 0, 3, 1]
pre = list(accumulate(dp, initial=0))
print(pre, "| Σ dp[0..3] =", pre[4])

[0, 0, 2, 2, 5, 6] | Σ dp[0..3] = 5


### 3.3. Hai con trỏ trên hai danh sách đã sắp

Với `A` (sắp tăng, có trọng số `w`) và `B` (sắp tăng), tính cho mỗi `b ∈ B`: `Σ_{a ∈ A, a ≤ b} w(a)`. Duyệt `B` tăng dần, con trỏ `p` trên `A` chỉ tiến, cộng dồn trọng số. Tổng `O(|A| + |B|)`.

In [5]:
A, w = [1, 4, 6, 9], [5, 1, 2, 7]
B = [0, 4, 5, 10]
p, acc, out = 0, 0, []
for b in B:
    while p < len(A) and A[p] <= b:
        acc += w[p]; p += 1
    out.append(acc)
print(out)

[0, 6, 6, 15]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mảng không giảm ⇒ khi chọn `arr[i]`, chỉ cần biết `arr[i-1]` (giá trị trước). Trạng thái tự nhiên: **(vị trí, giá trị cuối)**.

Tính tay `digitSum = [1, 1]`: ứng viên `{1, 10, 100, 1000}` cho cả hai vị trí, cần `arr[0] ≤ arr[1]` ⇒ số cặp không giảm từ 4 giá trị = `C(4+1, 2) = 10`.

Tính tay `digitSum = [2, 1]`: ứng viên tổng 2 `{2, 11, 20, 101, 110, 200, 1001, 1010, 1100, 2000}`; phần tử sau thuộc `{1, 10, 100, 1000}` và ≥ phần tử trước:

- `2 → 10, 100, 1000` (3); `11, 20 → 100, 1000` (2 + 2); `101, 110, 200 → 1000` (3 × 1); các số còn lại → 0. Tổng `3 + 4 + 3 = 10`.

## 5. Cách 1 — Brute Force: thử mọi tổ hợp ứng viên

`itertools.product` trên các danh sách ứng viên, đếm tổ hợp không giảm. Số tổ hợp `∏ |cand[s_i]|` — chỉ dùng với `n` rất nhỏ.

In [6]:
def brute(digitSum):
    lists = [cand[s] if s <= 50 else [] for s in digitSum]
    cnt = 0
    for combo in product(*lists):
        if all(combo[i] <= combo[i + 1] for i in range(len(combo) - 1)):
            cnt += 1
    return cnt % MOD

for ds, exp in examples:
    print(brute(ds), "kỳ vọng", exp)
print("[1,1] ->", brute([1, 1]), "| [2,1] ->", brute([2, 1]))

6 kỳ vọng 6
4 kỳ vọng 4
0 kỳ vọng 0
[1,1] -> 10 | [2,1] -> 10


**Nút thắt:** số tổ hợp tăng theo cấp số nhân. Nhưng đáp án chỉ phụ thuộc vào **giá trị cuối** ⇒ DP.

## 6. Ý tưởng thuật toán / bước đột phá

**DP:** `dp_i[v]` = số mảng hợp lệ cho `digitSum[0..i]` với `arr[i] = v` (chỉ khác 0 khi `v ∈ cand[digitSum[i]]`).

- `dp_0[v] = 1` với mọi `v ∈ cand[digitSum[0]]`.
- `dp_i[v] = Σ_{u ≤ v} dp_{i-1}[u]` với `v ∈ cand[digitSum[i]]`.
- Đáp án `= Σ_v dp_{n-1}[v]`.

**Đột phá 1 — tổng tiền tố:** `Σ_{u ≤ v}` là tổng tiền tố ⇒ mỗi tầng `O(V)` với `V = 5001`.

**Đột phá 2 — chỉ làm trên ứng viên:** `dp_{i-1}` chỉ khác 0 trên `cand[s_{i-1}]` (đã sắp). Với hai con trỏ, tầng `i` tốn `O(|cand[s_{i-1}]| + |cand[s_i]|)` ≤ vài trăm ⇒ tổng rất nhỏ.

**Quy tắc sớm:** nếu `digitSum[i] > 50` hoặc `cand[digitSum[i]]` rỗng (ví dụ tổng 49 hay 50 không đạt được trong `[0, 5000]`) ⇒ đáp án `0`.

## 7. Các cách tiếp theo

- **Cách 2:** DP thẳng `O(Σ |cand_{i-1}| · |cand_i|)` (vòng lặp lồng).
- **Cách 3:** DP trên toàn trục giá trị `0..5000` + tổng tiền tố — `O(n · V)`.
- **Cách 4:** DP trên ứng viên + **hai con trỏ** — `O(Σ |cand_i|)` (lời giải nộp chính).

### 7.1. Cách 2 — DP lồng hai vòng

In [7]:
def dp_nested(digitSum):
    if any(s > 50 or not cand[s] for s in digitSum):
        return 0
    prev_vals = cand[digitSum[0]]
    prev = [1] * len(prev_vals)
    for s in digitSum[1:]:
        cur_vals = cand[s]
        cur = []
        for v in cur_vals:
            total = 0
            for u, c in zip(prev_vals, prev):
                if u <= v:
                    total += c
            cur.append(total % MOD)
        prev_vals, prev = cur_vals, cur
    return sum(prev) % MOD

for ds, exp in examples:
    print(dp_nested(ds), "kỳ vọng", exp)

6 kỳ vọng 6
4 kỳ vọng 4
0 kỳ vọng 0


**Độ phức tạp:** `O(Σ |cand_{i-1}| · |cand_i|)` — tệ nhất `1000 · 300² ≈ 9·10^7` (chậm trong Python).

### 7.2. Cách 3 — DP trên trục giá trị + tổng tiền tố

**Giải thích:** giữ mảng `dp` dài `V = 5001`. Mỗi tầng: `pre = accumulate(dp)`, rồi `new[v] = pre[v+1]` cho `v ∈ cand[s]`, còn lại 0.

In [8]:
def dp_prefix(digitSum):
    if any(s > 50 or not cand[s] for s in digitSum):
        return 0
    V = LIMIT + 1
    dp = [0] * V
    for v in cand[digitSum[0]]:
        dp[v] = 1
    for s in digitSum[1:]:
        pre = list(accumulate(dp, initial=0))
        new = [0] * V
        for v in cand[s]:
            new[v] = pre[v + 1] % MOD
        dp = new
    return sum(dp) % MOD

for ds, exp in examples:
    print(dp_prefix(ds), "kỳ vọng", exp)

6 kỳ vọng 6
4 kỳ vọng 4
0 kỳ vọng 0


In [9]:
class SolutionPrefix:
    def countArrays(self, digitSum: list[int]) -> int:
        return dp_prefix(digitSum)

print([SolutionPrefix().countArrays(ds) for ds, _ in examples])

[6, 4, 0]


**Độ phức tạp:** `O(n · V)` = `5·10^6` thao tác (phần lớn trong `accumulate` chạy tốc độ C), bộ nhớ `O(V)`.

### 7.3. Cách 4 — DP trên ứng viên + hai con trỏ (lời giải chính)

**Giải thích từng đoạn:**

1. Kiểm tra mọi `digitSum[i]` có ứng viên; nếu không ⇒ `0`.
2. `prev_vals = cand[s_0]`, `prev = [1, …]`.
3. Tầng mới: duyệt `v` tăng dần trong `cand[s_i]`; con trỏ `p` trên `prev_vals` tiến khi `prev_vals[p] ≤ v`, cộng dồn `acc`. `cur[v] = acc`.

In [10]:
def two_pointer(digitSum):
    if any(s > 50 or not cand[s] for s in digitSum):
        return 0
    prev_vals = cand[digitSum[0]]
    prev = [1] * len(prev_vals)
    for s in digitSum[1:]:
        cur_vals = cand[s]
        cur = []
        p, acc = 0, 0
        for v in cur_vals:
            while p < len(prev_vals) and prev_vals[p] <= v:
                acc += prev[p]
                p += 1
            cur.append(acc % MOD)
        prev_vals, prev = cur_vals, cur
    return sum(prev) % MOD

for ds, exp in examples:
    print(two_pointer(ds), "kỳ vọng", exp)

6 kỳ vọng 6
4 kỳ vọng 4
0 kỳ vọng 0


**Độ phức tạp Cách 4:** tiền xử lý `O(V · log V)` (tính tổng chữ số), DP `O(Σ (|cand_{i-1}| + |cand_i|))` ≤ `2 · 1000 · 300` ⇒ rất nhanh; bộ nhớ `O(V)`.

## 8. TRACE từng bước

Trace Cách 4 với `digitSum = [2, 1]`.

- Tầng 0: `prev_vals = [2, 11, 20, 101, 110, 200, 1001, 1010, 1100, 2000]`, `prev = [1]*10`.
- Tầng 1: `cur_vals = [1, 10, 100, 1000]`:

| v | con trỏ tiến qua | acc = cur[v] |
|---|---|---|
| 1 | (không có `≤ 1`) | 0 |
| 10 | 2 | 1 |
| 100 | 11, 20 | 3 |
| 1000 | 101, 110, 200 | 6 |

Tổng `0 + 1 + 3 + 6 = 10` ✔ (khớp tính tay mục 4).

In [11]:
def trace(digitSum):
    prev_vals = cand[digitSum[0]]; prev = [1] * len(prev_vals)
    print("tầng 0:", list(zip(prev_vals, prev)))
    for i, s in enumerate(digitSum[1:], 1):
        cur_vals = cand[s]; cur = []; p = acc = 0
        for v in cur_vals:
            passed = []
            while p < len(prev_vals) and prev_vals[p] <= v:
                acc += prev[p]; passed.append(prev_vals[p]); p += 1
            cur.append(acc)
            print(f"  tầng {i}, v={v}: con trỏ qua {passed} -> dp={acc}")
        prev_vals, prev = cur_vals, cur
    print("đáp án =", sum(prev))

trace([2, 1])

tầng 0: [(2, 1), (11, 1), (20, 1), (101, 1), (110, 1), (200, 1), (1001, 1), (1010, 1), (1100, 1), (2000, 1)]
  tầng 1, v=1: con trỏ qua [] -> dp=0
  tầng 1, v=10: con trỏ qua [2] -> dp=1
  tầng 1, v=100: con trỏ qua [11, 20] -> dp=3
  tầng 1, v=1000: con trỏ qua [101, 110, 200] -> dp=6
đáp án = 10


In [12]:
trace([25, 1])

tầng 0: [(799, 1), (889, 1), (898, 1), (979, 1), (988, 1), (997, 1), (1699, 1), (1789, 1), (1798, 1), (1879, 1), (1888, 1), (1897, 1), (1969, 1), (1978, 1), (1987, 1), (1996, 1), (2599, 1), (2689, 1), (2698, 1), (2779, 1), (2788, 1), (2797, 1), (2869, 1), (2878, 1), (2887, 1), (2896, 1), (2959, 1), (2968, 1), (2977, 1), (2986, 1), (2995, 1), (3499, 1), (3589, 1), (3598, 1), (3679, 1), (3688, 1), (3697, 1), (3769, 1), (3778, 1), (3787, 1), (3796, 1), (3859, 1), (3868, 1), (3877, 1), (3886, 1), (3895, 1), (3949, 1), (3958, 1), (3967, 1), (3976, 1), (3985, 1), (3994, 1), (4399, 1), (4489, 1), (4498, 1), (4579, 1), (4588, 1), (4597, 1), (4669, 1), (4678, 1), (4687, 1), (4696, 1), (4759, 1), (4768, 1), (4777, 1), (4786, 1), (4795, 1), (4849, 1), (4858, 1), (4867, 1), (4876, 1), (4885, 1), (4894, 1), (4939, 1), (4948, 1), (4957, 1), (4966, 1), (4975, 1), (4984, 1), (4993, 1)]
  tầng 1, v=1: con trỏ qua [] -> dp=0
  tầng 1, v=10: con trỏ qua [] -> dp=0
  tầng 1, v=100: con trỏ qua [] -> dp=0


Trace thêm `digitSum = [1, 1, 1]`: mỗi tầng là "số mảng không giảm độ dài `i+1` lấy từ `{1, 10, 100, 1000}` kết thúc tại `v`". Tầng 1 cho `[1, 2, 3, 4]`, tầng 2 là tổng tiền tố của nó `[1, 3, 6, 10]` (số tam giác), tổng `20 = C(6, 3)` — đúng công thức tổ hợp lặp chọn 3 phần tử từ 4 giá trị.

In [13]:
trace([1, 1, 1])
from math import comb
print("C(4 + 3 - 1, 3) =", comb(6, 3))

tầng 0: [(1, 1), (10, 1), (100, 1), (1000, 1)]
  tầng 1, v=1: con trỏ qua [1] -> dp=1
  tầng 1, v=10: con trỏ qua [10] -> dp=2
  tầng 1, v=100: con trỏ qua [100] -> dp=3
  tầng 1, v=1000: con trỏ qua [1000] -> dp=4
  tầng 2, v=1: con trỏ qua [1] -> dp=1
  tầng 2, v=10: con trỏ qua [10] -> dp=3
  tầng 2, v=100: con trỏ qua [100] -> dp=6
  tầng 2, v=1000: con trỏ qua [1000] -> dp=10
đáp án = 20
C(4 + 3 - 1, 3) = 20


## 9. Code nộp LeetCode

In [14]:
class Solution:
    def countArrays(self, digitSum: list[int]) -> int:
        MOD = 10**9 + 7
        LIMIT = 5000
        cand = [[] for _ in range(51)]
        for x in range(LIMIT + 1):
            s, y = 0, x
            while y:
                s += y % 10
                y //= 10
            cand[s].append(x)
        for s in digitSum:
            if s > 50 or not cand[s]:
                return 0
        prev_vals = cand[digitSum[0]]
        prev = [1] * len(prev_vals)
        for s in digitSum[1:]:
            cur_vals = cand[s]
            cur = []
            p = acc = 0
            m = len(prev_vals)
            for v in cur_vals:
                while p < m and prev_vals[p] <= v:
                    acc += prev[p]
                    p += 1
                cur.append(acc % MOD)
            prev_vals, prev = cur_vals, cur
        return sum(prev) % MOD

sol = Solution()
for ds, exp in examples:
    got = sol.countArrays(ds)
    print(ds, "->", got, "✔" if got == exp else "✘")

[25, 1] -> 6 ✔
[1] -> 4 ✔
[2, 49, 23] -> 0 ✔


## 10. Vì sao thuật toán đúng

1. **Phân hoạch theo giá trị cuối:** mỗi mảng hợp lệ độ dài `i + 1` có đúng một giá trị cuối `v`; bỏ phần tử cuối được một mảng hợp lệ độ dài `i` với giá trị cuối `u ≤ v`. Ngược lại, nối `v ∈ cand[s_i]` vào mảng hợp lệ kết thúc bằng `u ≤ v` cho mảng hợp lệ. Song ánh này cho công thức `dp_i[v] = Σ_{u ≤ v} dp_{i-1}[u]`.
2. **Hai con trỏ:** vì `cur_vals` tăng, tập `{u ∈ prev_vals : u ≤ v}` là một **tiền tố** của `prev_vals` và chỉ mở rộng khi `v` tăng ⇒ `acc` đúng bằng tổng trên tiền tố đó.
3. **Modulo:** mọi phép là cộng nên lấy modulo bất kỳ lúc nào cũng đúng.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(digitSum)`, `V = 5001`, `c = max |cand[s]|`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `∏ |cand_i|` | `O(n)` |
| 2. DP lồng | `O(n c²)` | `O(c)` |
| 3. DP + tiền tố trên trục | `O(n V)` | `O(V)` |
| 4. DP + hai con trỏ | `O(V log V + n c)` | `O(V)` |

In [15]:
print("c = max |cand[s]| =", max(map(len, cand)), "| n·c tối đa ≈", 1000 * max(map(len, cand)))

c = max |cand[s]| = 365 | n·c tối đa ≈ 365000


**So sánh thực tế.** Cách 3 luôn xử lý đủ `5001` giá trị mỗi tầng (nhưng phần lớn trong `accumulate` ở tốc độ C), còn Cách 4 chỉ xử lý `|cand[s_{i-1}]| + |cand[s_i]| ≤ 730` phần tử mỗi tầng bằng vòng lặp Python. Với `n = 1000`, cả hai đều dưới một giây; Cách 2 (lồng hai vòng) có thể tới `1000 · 365² ≈ 1,3·10^8` phép so sánh — quá chậm ở ca xấu nhất. Đo nhanh dưới đây (dãy tổng chữ số nằm giữa phân bố, nơi số ứng viên lớn nhất):

In [16]:
ds_mid = [15, 16, 17, 18] * 50
for name, f in (("Cách 3 (trục)", dp_prefix), ("Cách 4 (hai con trỏ)", two_pointer)):
    t0 = time.time(); r = f(ds_mid); print(f"{name}: {r}, {time.time() - t0:.3f}s")

Cách 3 (trục): 348536269, 0.024s
Cách 4 (hai con trỏ): 348536269, 0.011s


## 12. Bộ test

In [17]:
def run_all(ds, with_brute=False):
    res = [dp_nested(ds), dp_prefix(ds), two_pointer(ds), Solution().countArrays(ds)]
    if with_brute:
        res.append(brute(ds))
    return res

for ds, exp in examples:
    assert all(v == exp for v in run_all(ds, True))
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [18]:
# 12.2 ca biên (tự dựng, kiểm chứng bằng vét cạn)
cases = [
    ([0], 1),                   # chỉ số 0
    ([0, 0, 0], 1),             # 0,0,0
    ([1, 1], 10),               # C(5, 2)
    ([2, 1], 10),
    ([50], 0),                  # không đạt được trong [0, 5000]
    ([31], None),               # 4999 có tổng 31 -> có ứng viên
    ([1, 0], 0),                # sau số dương không thể là 0
]
for ds, exp in cases:
    r = run_all(ds, with_brute=True)
    assert len(set(r)) == 1, (ds, r)
    if exp is not None:
        assert r[0] == exp, (ds, r)
    print(ds, "->", r[0])

[0] -> 1
[0, 0, 0] -> 1
[1, 1] -> 10
[2, 1] -> 10
[50] -> 0
[31] -> 1
[1, 0] -> 0


In [19]:
# 12.3 phản ví dụ cho ý tưởng sai "đếm độc lập rồi nhân" (quên điều kiện không giảm)
def wrong_product(ds):
    res = 1
    for s in ds:
        res = res * len(cand[s]) % MOD
    return res

print("sai:", wrong_product([25, 1]), "| đúng:", Solution().countArrays([25, 1]))

sai: 320 | đúng: 6


In [20]:
# 12.4 đối chiếu ngẫu nhiên
rng = random.Random(3883)
for _ in range(200):
    n = rng.randint(1, 3)
    ds = [rng.choice([0, 1, 2, 3, 27, 28, 30, 31, 35]) for _ in range(n)]
    r = run_all(ds, with_brute=(n <= 2 or max(len(cand[s]) for s in ds) <= 40))
    assert len(set(r)) == 1, (ds, r)
print("200 test ngẫu nhiên (có vét cạn khi khả thi): OK")
for _ in range(50):
    ds = [rng.randint(0, 30) for _ in range(rng.randint(1, 60))]
    r = run_all(ds)
    assert len(set(r)) == 1
print("50 test dài (so 4 cách): OK")

200 test ngẫu nhiên (có vét cạn khi khả thi): OK


50 test dài (so 4 cách): OK


In [21]:
# 12.5 test lớn: n = 1000
rng = random.Random(5)
ds = sorted(rng.randint(1, 28) for _ in range(1000))
t0 = time.time(); a = Solution().countArrays(ds); t1 = time.time()
b = dp_prefix(ds)
print("đáp án:", a, "| khớp Cách 3:", a == b, "| thời gian < 1s:", t1 - t0 < 1)
ds2 = [1] * 1000
print("1000 phần tử tổng chữ số 1 (chọn không giảm từ 4 giá trị = C(1003, 3)):", Solution().countArrays(ds2), (1003 * 1002 * 1001 // 6) % MOD)

đáp án: 254716176 | khớp Cách 3: True | thời gian < 1s: True
1000 phần tử tổng chữ số 1 (chọn không giảm từ 4 giá trị = C(1003, 3)): 167668501 167668501


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `<` thay vì `≤` (bắt buộc tăng ngặt)

In [22]:
def wrong_strict(ds):
    prev_vals = cand[ds[0]]; prev = [1] * len(prev_vals)
    for s in ds[1:]:
        cur = []
        for v in cand[s]:
            cur.append(sum(c for u, c in zip(prev_vals, prev) if u < v))    # SAI: phải là <=
        prev_vals, prev = cand[s], cur
    return sum(prev) % MOD

print("sai:", wrong_strict([1, 1]), "| đúng:", Solution().countArrays([1, 1]))

sai: 6 | đúng: 10


### Lỗi 2 — Quên giới hạn `5000` (sinh ứng viên tới 9999)

In [23]:
def wrong_limit(ds):
    cand2 = [[] for _ in range(51)]
    for x in range(10**4 + 1):                      # SAI: vượt 5000 (gồm cả 10000)
        cand2[digit_sum(x)].append(x)
    if any(not cand2[s] for s in ds): return 0
    prev_vals = cand2[ds[0]]; prev = [1] * len(prev_vals)
    for s in ds[1:]:
        cur = []; p = acc = 0
        for v in cand2[s]:
            while p < len(prev_vals) and prev_vals[p] <= v:
                acc += prev[p]; p += 1
            cur.append(acc)
        prev_vals, prev = cand2[s], cur
    return sum(prev) % MOD

print("sai:", wrong_limit([1]), "| đúng:", Solution().countArrays([1]))

sai: 5 | đúng: 4


### Lỗi 3 — Không xử lý tổng chữ số **không đạt được** (danh sách rỗng)

Nếu một `cand[s]` rỗng, DP sẽ tạo danh sách rỗng và các tầng sau tự ra 0 — đúng, nhưng nếu code truy cập `cand[s][0]` hoặc `max(cand[s])` sẽ **lỗi**. Kiểm tra sớm cho gọn.

In [24]:
try:
    print(max(cand[49]))
except ValueError as e:
    print("lỗi khi gọi max trên danh sách rỗng:", e)
print("kiểm tra sớm:", Solution().countArrays([2, 49, 23]))

lỗi khi gọi max trên danh sách rỗng: max() iterable argument is empty
kiểm tra sớm: 0


### Lỗi 4 — Quên modulo khi cộng dồn trên nhiều tầng

Python không tràn số nhưng số sẽ **rất lớn** (hàng trăm chữ số) làm chậm. Ở ngôn ngữ khác thì tràn. Lấy `% MOD` sau mỗi tầng.

In [25]:
def no_mod(ds):
    prev_vals = cand[ds[0]]; prev = [1] * len(prev_vals)
    for s in ds[1:]:
        cur = []; p = acc = 0
        for v in cand[s]:
            while p < len(prev_vals) and prev_vals[p] <= v:
                acc += prev[p]; p += 1
            cur.append(acc)                        # không mod
        prev_vals, prev = cand[s], cur
    return sum(prev)

big = no_mod([1] * 300 + [10] * 300)
print("số chữ số khi không mod:", len(str(big)), "| có mod:", Solution().countArrays([1] * 300 + [10] * 300), "| khớp:", big % MOD == Solution().countArrays([1] * 300 + [10] * 300))

số chữ số khi không mod: 158 | có mod: 969874769 | khớp: True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Đếm dãy không giảm, mỗi vị trí chọn từ một tập"** ⇒ DP theo giá trị cuối + **tổng tiền tố** (hoặc hai con trỏ khi tập đã sắp).
- **Giá trị nhỏ (≤ 5000)** ⇒ có thể tiền xử lý mọi số; **tổng chữ số ≤ 50** ⇒ nhóm theo tổng chữ số.
- Khi `dp` thưa (chỉ khác 0 trên ít điểm) ⇒ làm việc trên danh sách điểm thay vì cả trục.
- Liên quan: *Count the Number of Inversions*-style DP, *Number of Ways to Form a Target String* (LC 1639), *Find the Count of Monotonic Pairs* (LC 3250/3251).

## 15. Biến thể của bài

1. **Tăng ngặt**: thay `≤` bằng `<` (con trỏ tiến khi `prev_vals[p] < v`).
2. **Không tăng**: duyệt ngược hoặc đổi dấu.
3. **Giới hạn khác cho từng vị trí** (`arr[i] ≤ limit[i]`): lọc `cand` theo `limit[i]`.
4. **Ràng buộc "tích chữ số"** hoặc "số chữ số" thay vì tổng chữ số: chỉ đổi cách nhóm ứng viên.
5. **Chênh lệch kề nhau ≤ d**: tổng trên cửa sổ `[v - d, v]` ⇒ hai con trỏ cho cả hai đầu.

In [26]:
# Biến thể 5: thêm điều kiện arr[i] - arr[i-1] <= d
def with_gap(ds, d):
    if any(not cand[s] for s in ds): return 0
    prev_vals = cand[ds[0]]; prev = [1] * len(prev_vals)
    for s in ds[1:]:
        pre = list(accumulate(prev, initial=0)); cur = []
        for v in cand[s]:
            hi = bisect_right(prev_vals, v)                 # u <= v
            lo = bisect_right(prev_vals, v - d - 1)         # u >= v - d
            cur.append((pre[hi] - pre[lo]) % MOD)
        prev_vals, prev = cand[s], cur
    return sum(prev) % MOD

def brute_gap(ds, d):
    return sum(1 for c in product(*[cand[s] for s in ds]) if all(0 <= c[i + 1] - c[i] <= d for i in range(len(c) - 1)))

for ds, d in [([1, 1], 50), ([2, 1], 100), ([3, 2, 2], 300), ([4, 4], 0)]:
    assert with_gap(ds, d) == brute_gap(ds, d)
print("Biến thể 5 khớp vét cạn.")

Biến thể 5 khớp vét cạn.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Thử mọi tổ hợp | mũ | `O(n)` | Kiểm thử |
| 2. DP lồng | `Σ_{u ≤ v}` bằng vòng lặp | `O(n c²)` | `O(c)` | Dễ hiểu |
| 3. DP + tiền tố trục | `accumulate` trên `V` giá trị | `O(n V)` | `O(V)` | Khi muốn code ngắn |
| 4. DP + hai con trỏ | Chỉ trên ứng viên | `O(n c)` | `O(V)` | **Lời giải chuẩn** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countArrays([0, 1])` = ?

**Bài 2 (tính tay).** `countArrays([3])` — liệt kê các số `≤ 5000` có tổng chữ số 3 (gợi ý: số chữ số ≤ 4).

**Bài 3 (điền TODO).** Viết `count_strict(digitSum)` cho biến thể **tăng ngặt**.

**Bài 4 (tự viết lại).** Viết lại Cách 4 dùng `bisect_right` + tổng tiền tố trên `prev` (không dùng hai con trỏ).

**Bài 5 (mini challenge).** Mỗi vị trí có thêm giới hạn trên `limit[i]` (`arr[i] ≤ limit[i]`). Viết lời giải và kiểm chứng.

In [27]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_strict(digitSum):
    # TODO: như two_pointer nhưng con trỏ tiến khi prev_vals[p] < v
    pass

In [28]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_with_limits(digitSum, limit):
    # TODO: lọc cand[s] theo <= limit[i] trước khi DP
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Phần tử đầu là `0`. (2) Phần tử sau ∈ `{1, 10, 100, 1000}` đều ≥ 0. (3) Đáp án `4`.

**Bài 2.** (1) Số `≤ 5000` tối đa 4 chữ số; đếm nghiệm `a+b+c+d = 3` với chữ số. (2) `C(6, 3) = 20` bộ (đều < 5000 vì chữ số đầu ≤ 3). (3) Đáp án `20`.

**Bài 3.** (1) Đổi `<=` thành `<`. (2) `[1, 1]` cho `C(4, 2) = 6`. (3) Đối chiếu vét cạn.

**Bài 4.** (1) `pre = accumulate(prev, initial=0)`. (2) `idx = bisect_right(prev_vals, v)`. (3) `cur = pre[idx]`.

**Bài 5.** (1) `cand_i = [x for x in cand[s] if x <= limit[i]]`. (2) Rỗng ⇒ 0. (3) Phần còn lại như Cách 4.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1, 2
assert Solution().countArrays([0, 1]) == 4
assert Solution().countArrays([3]) == 20 == len(cand[3])
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [30]:
# Bài 3
def count_strict_sol(ds):
    if any(not cand[s] for s in ds): return 0
    prev_vals = cand[ds[0]]; prev = [1] * len(prev_vals)
    for s in ds[1:]:
        cur = []; p = acc = 0
        for v in cand[s]:
            while p < len(prev_vals) and prev_vals[p] < v:
                acc += prev[p]; p += 1
            cur.append(acc % MOD)
        prev_vals, prev = cand[s], cur
    return sum(prev) % MOD

assert count_strict_sol([1, 1]) == 6
assert count_strict_sol([2, 1, 3]) == sum(1 for c in product(cand[2], cand[1], cand[3]) if c[0] < c[1] < c[2])
print("Bài 3 OK")

Bài 3 OK


In [31]:
# Bài 4
def sol_bisect(ds):
    if any(not cand[s] for s in ds): return 0
    prev_vals = cand[ds[0]]; prev = [1] * len(prev_vals)
    for s in ds[1:]:
        pre = list(accumulate(prev, initial=0))
        prev, prev_vals = [pre[bisect_right(prev_vals, v)] % MOD for v in cand[s]], cand[s]
    return sum(prev) % MOD

rng = random.Random(4)
for _ in range(100):
    ds = [rng.randint(0, 35) for _ in range(rng.randint(1, 20))]
    assert sol_bisect(ds) == Solution().countArrays(ds)
print("Bài 4 OK")

Bài 4 OK


In [32]:
# Bài 5
def count_with_limits_sol(ds, limit):
    lists = [[x for x in cand[s] if x <= lim] for s, lim in zip(ds, limit)]
    if any(not l for l in lists): return 0
    prev_vals = lists[0]; prev = [1] * len(prev_vals)
    for cur_vals in lists[1:]:
        cur = []; p = acc = 0
        for v in cur_vals:
            while p < len(prev_vals) and prev_vals[p] <= v:
                acc += prev[p]; p += 1
            cur.append(acc % MOD)
        prev_vals, prev = cur_vals, cur
    return sum(prev) % MOD

def brute_limits(ds, limit):
    lists = [[x for x in cand[s] if x <= lim] for s, lim in zip(ds, limit)]
    return sum(1 for c in product(*lists) if all(c[i] <= c[i + 1] for i in range(len(c) - 1)))

rng = random.Random(5)
for _ in range(100):
    n = rng.randint(1, 3); ds = [rng.randint(0, 4) for _ in range(n)]; lim = [rng.randint(0, 5000) for _ in range(n)]
    assert count_with_limits_sol(ds, lim) == brute_limits(ds, lim)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cand[s] = sorted các x in [0, 5000] có digit_sum(x) == s
if any(not cand[s] for s in digitSum): return 0
prev_vals, prev = cand[ds[0]], [1]*len(cand[ds[0]])
for s in ds[1:]:
    p = acc = 0; cur = []
    for v in cand[s]:
        while p < len(prev_vals) and prev_vals[p] <= v:
            acc += prev[p]; p += 1
        cur.append(acc % MOD)
    prev_vals, prev = cand[s], cur
return sum(prev) % MOD
```

**Checklist:**

- [ ] Ứng viên giới hạn đúng `[0, 5000]` và gồm cả `0` (tổng chữ số 0)?
- [ ] So sánh `≤` (không giảm), không phải `<`?
- [ ] Kiểm tra sớm tổng chữ số không đạt được (danh sách rỗng)?
- [ ] Lấy `% MOD` mỗi tầng?
- [ ] Danh sách ứng viên **đã sắp tăng** trước khi dùng hai con trỏ?